In [ ]:
# Install Dependencies
!pip -q install kaggle timm wandb tensorboard onnx onnxruntime onnxscript

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Setup output path in Google Drive
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/DR_Severity-Ensemble/")
OUTPUT_ROOT = DRIVE_ROOT / "outputs_dr"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Drive output root:", OUTPUT_ROOT)

In [ ]:
# Upload kaggle.json to access datasets in kaggle
from google.colab import files
files.upload()

In [ ]:
# Download dataset to local runtime
import shutil
from pathlib import Path
import os

KAGGLE_DIR = Path("/root/.kaggle")
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

if Path("/content/kaggle.json").exists():
    shutil.move("/content/kaggle.json", str(KAGGLE_DIR / "kaggle.json"))

os.chmod(str(KAGGLE_DIR / "kaggle.json"), 0o600)

LOCAL_ZIP_DIR = Path("/content/kaggle_data")
LOCAL_UNZIP_DIR = Path("/content/kaggle_data_unzipped")
LOCAL_ZIP_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_UNZIP_DIR.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy -p /content/kaggle_data
!unzip -q /content/kaggle_data/*.zip -d /content/kaggle_data_unzipped

print("Dataset downloaded to local runtime.")

In [ ]:
# Root Datasets
from pathlib import Path

base = Path("/content/kaggle_data_unzipped")

candidates = []
for p in base.rglob("*"):
    if p.is_dir():
        children = sorted([x.name for x in p.iterdir() if x.is_dir()])
        if set(["train", "val", "test"]).issubset(set(children)):
            candidates.append(str(p))

print("Candidate dataset roots:")
for c in candidates:
    print(c)

DATA_ROOT = "/content/kaggle_data_unzipped/augmented_resized_V2"
print("Using DATA_ROOT =", DATA_ROOT)

In [ ]:
# Config
import os
import re
import gc
import json
import time
import copy
import math
import random
import warnings
from pathlib import Path
from collections import Counter


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

import torchvision
from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_large, MobileNet_V3_Large_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    swin_t, Swin_T_Weights
)

import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
    balanced_accuracy_score
)

try:
    import wandb
    WANDB_AVAILABLE = True
except:
    WANDB_AVAILABLE = False

SEED = 42
NUM_CLASSES = 5
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

In [ ]:
# Scanning dataset and building metadata
def parse_label_from_path(path: Path):
    for parent in path.parents:
        if parent.name in {"0", "1", "2", "3", "4"}:
            return int(parent.name)
    return None

def extract_group_id(path: Path):
    """
    Heuristik grouping untuk gambar asli + seluruh augmentasinya.
    """
    stem = path.stem.lower()

    patterns = [
        r"([_-]aug\d+)$",
        r"([_-]v\d+)$",
        r"([_-](rotate|rot)\d*)$",
        r"([_-]flip(ped)?)$",
        r"([_-](blur|bright|brightness|contrast|crop|zoom|gamma|noise|sat|hue|clahe|gaussian))$",
        r"([_-]copy\d*)$",
        r"([_-]variant\d*)$",
    ]

    changed = True
    while changed:
        changed = False
        for pat in patterns:
            new_stem = re.sub(pat, "", stem)
            if new_stem != stem:
                stem = new_stem
                changed = True

    stem = re.sub(r"[_-]+$", "", stem)
    return stem if stem else path.stem.lower()

def scan_dataset(data_root):
    rows = []
    for p in Path(data_root).rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTS:
            label = parse_label_from_path(p)
            if label is None:
                continue
            rows.append({
                "filepath": str(p.resolve()),
                "filename": p.name,
                "label": label,
                "group_id": extract_group_id(p),
                "original_split_folder": next((x.name for x in p.parents if x.name in {"train", "val", "test"}), "unknown")
            })
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("Dataset tidak ditemukan atau struktur folder tidak sesuai.")
    return df

df_all = scan_dataset(DATA_ROOT)
print(df_all.head())
print("Total images:", len(df_all))
print(df_all["label"].value_counts().sort_index())

In [ ]:
# Dataset distribution analysis
dist = df_all["label"].value_counts().sort_index()
dist_df = pd.DataFrame({
    "label": dist.index,
    "class_name": [CLASS_NAMES[i] for i in dist.index],
    "count": dist.values,
    "percent": (dist.values / dist.values.sum()) * 100
})
display(dist_df)

plt.figure(figsize=(8,5))
sns.barplot(data=dist_df, x="class_name", y="count")
plt.xticks(rotation=20)
plt.title("Class Distribution (Image Level)")
plt.tight_layout()
plt.show()

group_dist = df_all.groupby("group_id")["label"].first().value_counts().sort_index()
group_dist_df = pd.DataFrame({
    "label": group_dist.index,
    "class_name": [CLASS_NAMES[i] for i in group_dist.index],
    "group_count": group_dist.values
})
display(group_dist_df)

plt.figure(figsize=(8,5))
sns.barplot(data=group_dist_df, x="class_name", y="group_count")
plt.xticks(rotation=20)
plt.title("Class Distribution (Group Level)")
plt.tight_layout()
plt.show()

In [ ]:
# Dataset Splitting (Group base) 70/15/15
def group_table(df):
    return (
        df.groupby("group_id")
        .agg(label=("label", "first"), n_images=("filepath", "count"))
        .reset_index()
    )

groups_df = group_table(df_all)
print("Unique groups:", len(groups_df))

def stratified_group_split_70_15_15(groups_df, seed=42):
    X = np.arange(len(groups_df))
    y = groups_df["label"].values
    groups = groups_df["group_id"].values

    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    outer_candidates = list(outer.split(X, y, groups))

    best_outer = None
    best_gap = 1e9
    for tr_idx, te_idx in outer_candidates:
        frac = len(te_idx) / len(X)
        gap = abs(frac - 0.15)
        if gap < best_gap:
            best_gap = gap
            best_outer = (tr_idx, te_idx)

    trainval_idx, test_idx = best_outer

    rem = groups_df.iloc[trainval_idx].reset_index(drop=True)
    X2 = np.arange(len(rem))
    y2 = rem["label"].values
    g2 = rem["group_id"].values

    inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed + 1)
    inner_candidates = list(inner.split(X2, y2, g2))

    best_inner = None
    best_gap = 1e9
    target_val_ratio = 0.15 / 0.85
    for tr2_idx, va_idx in inner_candidates:
        frac = len(va_idx) / len(X2)
        gap = abs(frac - target_val_ratio)
        if gap < best_gap:
            best_gap = gap
            best_inner = (tr2_idx, va_idx)

    train_idx2, val_idx2 = best_inner

    train_groups = set(rem.iloc[train_idx2]["group_id"].tolist())
    val_groups = set(rem.iloc[val_idx2]["group_id"].tolist())
    test_groups = set(groups_df.iloc[test_idx]["group_id"].tolist())

    return train_groups, val_groups, test_groups

train_groups, val_groups, test_groups = stratified_group_split_70_15_15(groups_df, seed=SEED)

df_all["split"] = df_all["group_id"].map(
    lambda x: "train" if x in train_groups else ("val" if x in val_groups else "test")
)

print(df_all["split"].value_counts())
print("Overlap train-val:", len(train_groups & val_groups))
print("Overlap train-test:", len(train_groups & test_groups))
print("Overlap val-test:", len(val_groups & test_groups))

split_csv = OUTPUT_ROOT / "split_metadata.csv"
df_all.to_csv(split_csv, index=False)

split_summary = {
    "total_images": int(len(df_all)),
    "total_groups": int(df_all["group_id"].nunique()),
    "split_counts": df_all["split"].value_counts().to_dict(),
    "label_counts_image_level": df_all["label"].value_counts().sort_index().to_dict(),
    "label_counts_group_level": groups_df["label"].value_counts().sort_index().to_dict(),
    "group_overlap": {
        "train_val": len(train_groups & val_groups),
        "train_test": len(train_groups & test_groups),
        "val_test": len(val_groups & test_groups),
    }
}

with open(OUTPUT_ROOT / "split_summary.json", "w") as f:
    json.dump(split_summary, f, indent=2)

print("Saved split metadata to:", split_csv)

In [ ]:
# Transform and dataset
class RetinoDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        label = int(row["label"])
        if self.transform:
            image = self.transform(image)
        return image, label, row["filepath"]

def get_transforms(img_size=224):
    train_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(12),
        transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        transforms.RandomErasing(p=0.1, scale=(0.02, 0.08), ratio=(0.3, 3.3), value="random")
    ])

    eval_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    return train_tfms, eval_tfms

In [ ]:
# Dataloader
def make_loaders(df, img_size=224, batch_size=16, num_workers=1, imbalance_mode="weighted_loss"):
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df   = df[df["split"] == "val"].reset_index(drop=True)
    test_df  = df[df["split"] == "test"].reset_index(drop=True)

    train_tfms, eval_tfms = get_transforms(img_size)

    train_ds = RetinoDataset(train_df, train_tfms)
    val_ds   = RetinoDataset(val_df, eval_tfms)
    test_ds  = RetinoDataset(test_df, eval_tfms)

    class_counts = train_df["label"].value_counts().sort_index()
    class_weights = torch.tensor(
        [len(train_df) / max(1, class_counts.get(i, 0)) for i in range(NUM_CLASSES)],
        dtype=torch.float
    )
    class_weights = class_weights / class_weights.mean()

    sampler = None
    shuffle = True
    if imbalance_mode == "oversample":
        sample_weights = train_df["label"].map(lambda x: class_weights[int(x)].item()).values
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )
        shuffle = False

    train_loader = DataLoader(
        train_ds, batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        num_workers=num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size,
        shuffle=False, num_workers=num_workers, pin_memory=True
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size,
        shuffle=False, num_workers=num_workers, pin_memory=True
    )

    return train_loader, val_loader, test_loader, class_weights, train_df, val_df, test_df

In [ ]:
# Build Model
class EffB0SwinEnsemble(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3):
        super().__init__()

        self.eff = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        eff_in = self.eff.classifier[1].in_features
        self.eff.classifier = nn.Identity()

        self.swin = swin_t(weights=Swin_T_Weights.DEFAULT)
        swin_in = self.swin.head.in_features
        self.swin.head = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Linear(eff_in + swin_in, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f1 = self.eff(x)
        f2 = self.swin(x)
        x = torch.cat([f1, f2], dim=1)
        return self.classifier(x)

def build_model(model_name, num_classes=5, dropout=0.3):

    # if model_name == "mobilenetv3":
    #     model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
    #     in_features = model.classifier[0].in_features
    #     model.classifier = nn.Sequential(
    #         nn.Linear(in_features, 512),
    #         nn.ReLU(),
    #         nn.Dropout(dropout),
    #         nn.Linear(512, num_classes)
    #     )

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    elif model_name == "swin_t":
        model = swin_t(weights=Swin_T_Weights.DEFAULT)
        in_features = model.head.in_features
        model.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    elif model_name == "effb0_swin_ensemble":
        model = EffB0SwinEnsemble(num_classes=num_classes, dropout=dropout)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model

In [ ]:
# Freeze model
def freeze_stage1(model, model_name):
    for p in model.parameters():
        p.requires_grad = False

    if model_name == "mobilenetv3":
        for p in model.classifier.parameters():
            p.requires_grad = True

    elif model_name == "efficientnet_b0":
        for p in model.classifier.parameters():
            p.requires_grad = True

    elif model_name == "swin_t":
        for p in model.head.parameters():
            p.requires_grad = True

    elif model_name == "effb0_swin_ensemble":
        for p in model.classifier.parameters():
            p.requires_grad = True

def unfreeze_stage2(model, model_name, mode="last_blocks"):
    for p in model.parameters():
        p.requires_grad = False

    if model_name == "mobilenetv3":
        for p in model.classifier.parameters():
            p.requires_grad = True
        if mode == "all":
            for p in model.parameters():
                p.requires_grad = True
        else:
            n = len(model.features)
            for i in range(max(0, int(n * 0.75)), n):
                for p in model.features[i].parameters():
                    p.requires_grad = True

    elif model_name == "efficientnet_b0":
        for p in model.classifier.parameters():
            p.requires_grad = True
        if mode == "all":
            for p in model.parameters():
                p.requires_grad = True
        else:
            n = len(model.features)
            for i in range(max(0, int(n * 0.7)), n):
                for p in model.features[i].parameters():
                    p.requires_grad = True

    elif model_name == "swin_t":
        for p in model.head.parameters():
            p.requires_grad = True
        if mode == "all":
            for p in model.parameters():
                p.requires_grad = True
        else:
            n = len(model.features)
            for i in range(max(0, n - 2), n):
                for p in model.features[i].parameters():
                    p.requires_grad = True
            if hasattr(model, "norm"):
                for p in model.norm.parameters():
                    p.requires_grad = True

    elif model_name == "effb0_swin_ensemble":
        for p in model.classifier.parameters():
            p.requires_grad = True
        if mode == "all":
            for p in model.parameters():
                p.requires_grad = True
        else:
            eff_n = len(model.eff.features)
            for i in range(max(0, int(eff_n * 0.7)), eff_n):
                for p in model.eff.features[i].parameters():
                    p.requires_grad = True

            swin_n = len(model.swin.features)
            for i in range(max(0, swin_n - 2), swin_n):
                for p in model.swin.features[i].parameters():
                    p.requires_grad = True

            if hasattr(model.swin, "norm"):
                for p in model.swin.norm.parameters():
                    p.requires_grad = True

In [ ]:
# Early Stopping
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = None
        self.counter = 0
        self.stop = False

    def step(self, metric):
        if self.best is None:
            self.best = metric
            return False

        improved = metric > (self.best + self.min_delta) if self.mode == "max" else metric < (self.best - self.min_delta)

        if improved:
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

        return self.stop

In [ ]:
# Run Epoch
from tqdm.auto import tqdm

def run_epoch(model, loader, criterion, optimizer=None, train=True, amp=True, epoch_desc=""):
    model.train() if train else model.eval()

    scaler = torch.cuda.amp.GradScaler(enabled=(amp and train and DEVICE.type == "cuda"))

    running_loss = 0.0
    y_true, y_pred, y_prob, paths_all = [], [], [], []

    pbar = tqdm(loader, desc=epoch_desc, leave=False)

    for images, labels, paths in pbar:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(amp and DEVICE.type == "cuda")):
                logits = model(images)
                loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())
        y_prob.extend(probs.detach().cpu().numpy().tolist())
        paths_all.extend(paths)

        # update progress bar
        current_loss = running_loss / max(1, len(y_true))
        current_acc = accuracy_score(y_true, y_pred) if len(y_true) > 0 else 0.0
        pbar.set_postfix({
            "loss": f"{current_loss:.4f}",
            "acc": f"{current_acc:.4f}"
        })

    avg_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    bacc = balanced_accuracy_score(y_true, y_pred)

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "qwk": qwk,
        "balanced_accuracy": bacc,
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "y_prob": np.array(y_prob),
        "paths": paths_all
    }

In [ ]:
# Train model
def train_one_model(
    model_name="mobilenetv3",
    img_size=224,
    batch_size=16,
    num_workers=2,
    epochs_head=5,
    epochs_ft=10,
    lr_head=1e-3,
    lr_ft=2e-4,
    weight_decay=1e-4,
    dropout=0.35,
    label_smoothing=0.05,
    imbalance_mode="weighted_loss",
    unfreeze_mode="last_blocks",
    use_wandb=False,
    wandb_project="dr-severity-colab",
    amp=True
):
    import matplotlib.pyplot as plt

    run_dir = OUTPUT_ROOT / model_name
    artifact_dir = run_dir / "artifacts"
    plot_dir = run_dir / "plots"
    log_dir = run_dir / "logs"
    pred_dir = run_dir / "predictions"

    artifact_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)
    pred_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print(f"START TRAINING: {model_name}")
    print(f"Output directory : {run_dir}")
    print(f"Device           : {DEVICE}")
    print(f"Image size       : {img_size}")
    print(f"Batch size       : {batch_size}")
    print(f"Epochs head      : {epochs_head}")
    print(f"Epochs finetune  : {epochs_ft}")
    print(f"Imbalance mode   : {imbalance_mode}")
    print("=" * 80)

    train_loader, val_loader, test_loader, class_weights, train_df, val_df, test_df = make_loaders(
        df_all,
        img_size=img_size,
        batch_size=batch_size,
        num_workers=num_workers,
        imbalance_mode=imbalance_mode
    )

    # TensorBoard - Sample Images
    try:
        import torchvision
        images, labels = next(iter(val_loader))
        grid = torchvision.utils.make_grid(images[:8])
        tb_writer.add_image("Sample Images", grid, 0)
    except Exception as e:
        print("TensorBoard image logging failed:", e)

    print(f"Train samples    : {len(train_df)}")
    print(f"Val samples      : {len(val_df)}")
    print(f"Test samples     : {len(test_df)}")
    print(f"Class weights    : {class_weights.tolist()}")
    print("-" * 80)

    model = build_model(model_name, num_classes=NUM_CLASSES, dropout=dropout).to(DEVICE)

    # TensorBoard - Model Graph
    try:
        dummy = torch.randn(1, 3, img_size, img_size).to(DEVICE)
        model.eval()
        torch.onnx.export(...)
    except Exception as e:
        print("ONNX export failed:", e)

    config_dict = {
        "model_name": model_name,
        "img_size": img_size,
        "batch_size": batch_size,
        "num_workers": num_workers,
        "epochs_head": epochs_head,
        "epochs_ft": epochs_ft,
        "lr_head": lr_head,
        "lr_ft": lr_ft,
        "weight_decay": weight_decay,
        "dropout": dropout,
        "label_smoothing": label_smoothing,
        "imbalance_mode": imbalance_mode,
        "unfreeze_mode": unfreeze_mode,
        "device": str(DEVICE),
        "train_size": len(train_df),
        "val_size": len(val_df),
        "test_size": len(test_df)
    }

    with open(run_dir / "config.json", "w") as f:
        json.dump(config_dict, f, indent=2)

    if imbalance_mode == "weighted_loss":
        with open(run_dir / "class_weights.json", "w") as f:
            json.dump({"class_weights": class_weights.tolist()}, f, indent=2)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), label_smoothing=label_smoothing)
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    tb_writer = SummaryWriter(log_dir=str(log_dir / "tensorboard"))

    if use_wandb and WANDB_AVAILABLE:
        wandb.init(
            project=wandb_project,
            name=model_name,
            dir="/content",
            config=config_dict
        )

    history = []
    best_state = None
    best_val_f1 = -1
    best_epoch = -1

    # =========================
    # Stage 1: Head training
    # =========================
    print("\n" + "=" * 80)
    print(f"STAGE 1: HEAD TRAINING ({model_name})")
    print("=" * 80)

    freeze_stage1(model, model_name)
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr_head, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    es = EarlyStopping(patience=3, min_delta=1e-4, mode="max")

    global_epoch = 0

    for epoch in range(epochs_head):
        global_epoch += 1
        epoch_start = time.time()

        tr = run_epoch(
            model, train_loader, criterion, optimizer=optimizer, train=True, amp=amp,
            epoch_desc=f"[HEAD] Epoch {epoch+1}/{epochs_head} - Train"
        )
        va = run_epoch(
            model, val_loader, criterion, optimizer=None, train=False, amp=amp,
            epoch_desc=f"[HEAD] Epoch {epoch+1}/{epochs_head} - Val"
        )

        scheduler.step(va["f1"])
        current_lr = optimizer.param_groups[0]["lr"]
        epoch_time = time.time() - epoch_start

        row = {
            "epoch": global_epoch,
            "stage": "head",
            "train_loss": tr["loss"],
            "val_loss": va["loss"],
            "train_acc": tr["accuracy"],
            "val_acc": va["accuracy"],
            "train_precision": tr["precision"],
            "val_precision": va["precision"],
            "train_recall": tr["recall"],
            "val_recall": va["recall"],
            "train_f1": tr["f1"],
            "val_f1": va["f1"],
            "train_qwk": tr["qwk"],
            "val_qwk": va["qwk"],
            "train_balanced_accuracy": tr["balanced_accuracy"],
            "val_balanced_accuracy": va["balanced_accuracy"],
            "lr": current_lr,
            "epoch_time_sec": epoch_time
        }
        history.append(row)

        for k, v in row.items():
            if k not in ["epoch", "stage"]:
                if "loss" in k:
                    tb_writer.add_scalar(f"Loss/{k}", v, global_epoch)
                elif "f1" in k or "qwk" in k or "accuracy" in k:
                    tb_writer.add_scalar(f"Metrics/{k}", v, global_epoch)
                elif "lr" in k:
                    tb_writer.add_scalar("LR", v, global_epoch)
                else:
                    tb_writer.add_scalar(f"Other/{k}", v, global_epoch)
        # TensorBoard - Weight Histogram
        if global_epoch % 2 == 0:  # not too heavy
            for name, param in model.named_parameters():
                if param.requires_grad:
                    tb_writer.add_histogram(name, param, global_epoch)

        if use_wandb and WANDB_AVAILABLE:
            wandb.log(row)

        print(
            f"[HEAD][Epoch {epoch+1:02d}/{epochs_head}] "
            f"time={epoch_time:.1f}s | "
            f"lr={current_lr:.6f} | "
            f"train_loss={tr['loss']:.4f} | val_loss={va['loss']:.4f} | "
            f"train_acc={tr['accuracy']:.4f} | val_acc={va['accuracy']:.4f} | "
            f"train_f1={tr['f1']:.4f} | val_f1={va['f1']:.4f} | "
            f"train_qwk={tr['qwk']:.4f} | val_qwk={va['qwk']:.4f}"
        )

        if va["f1"] > best_val_f1:
            best_val_f1 = va["f1"]
            best_epoch = global_epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, artifact_dir / "best_model_stage1.pth")
            print(f"  -> New best model saved at epoch {global_epoch} (val_f1={best_val_f1:.4f})")

        if es.step(va["f1"]):
            print("  -> Early stopping triggered in stage 1")
            break

    # =========================
    # Stage 2: Fine-tuning
    # =========================
    print("\n" + "=" * 80)
    print(f"STAGE 2: FINE-TUNING ({model_name})")
    print("=" * 80)

    unfreeze_stage2(model, model_name, mode=unfreeze_mode)
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr_ft, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    es = EarlyStopping(patience=5, min_delta=1e-4, mode="max")

    for epoch in range(epochs_ft):
        global_epoch += 1
        epoch_start = time.time()

        tr = run_epoch(
            model, train_loader, criterion, optimizer=optimizer, train=True, amp=amp,
            epoch_desc=f"[FT] Epoch {epoch+1}/{epochs_ft} - Train"
        )
        va = run_epoch(
            model, val_loader, criterion, optimizer=None, train=False, amp=amp,
            epoch_desc=f"[FT] Epoch {epoch+1}/{epochs_ft} - Val"
        )

        scheduler.step(va["f1"])
        current_lr = optimizer.param_groups[0]["lr"]
        epoch_time = time.time() - epoch_start

        row = {
            "epoch": global_epoch,
            "stage": "finetune",
            "train_loss": tr["loss"],
            "val_loss": va["loss"],
            "train_acc": tr["accuracy"],
            "val_acc": va["accuracy"],
            "train_precision": tr["precision"],
            "val_precision": va["precision"],
            "train_recall": tr["recall"],
            "val_recall": va["recall"],
            "train_f1": tr["f1"],
            "val_f1": va["f1"],
            "train_qwk": tr["qwk"],
            "val_qwk": va["qwk"],
            "train_balanced_accuracy": tr["balanced_accuracy"],
            "val_balanced_accuracy": va["balanced_accuracy"],
            "lr": current_lr,
            "epoch_time_sec": epoch_time
        }
        history.append(row)

        for k, v in row.items():
            if k not in ["epoch", "stage"]:
                if "loss" in k:
                    tb_writer.add_scalar(f"Loss/{k}", v, global_epoch)
                elif "f1" in k or "qwk" in k or "accuracy" in k:
                    tb_writer.add_scalar(f"Metrics/{k}", v, global_epoch)
                elif "lr" in k:
                    tb_writer.add_scalar("LR", v, global_epoch)
                else:
                    tb_writer.add_scalar(f"Other/{k}", v, global_epoch)

        # TensorBoard - Weight Histogram
        if global_epoch % 2 == 0:  # supaya tidak terlalu berat
            for name, param in model.named_parameters():
                if param.requires_grad:
                    tb_writer.add_histogram(name, param, global_epoch)

        if use_wandb and WANDB_AVAILABLE:
            wandb.log(row)

        print(
            f"[FT][Epoch {epoch+1:02d}/{epochs_ft}] "
            f"time={epoch_time:.1f}s | "
            f"lr={current_lr:.6f} | "
            f"train_loss={tr['loss']:.4f} | val_loss={va['loss']:.4f} | "
            f"train_acc={tr['accuracy']:.4f} | val_acc={va['accuracy']:.4f} | "
            f"train_f1={tr['f1']:.4f} | val_f1={va['f1']:.4f} | "
            f"train_qwk={tr['qwk']:.4f} | val_qwk={va['qwk']:.4f}"
        )

        if va["f1"] > best_val_f1:
            best_val_f1 = va["f1"]
            best_epoch = global_epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, artifact_dir / "best_model.pth")
            print(f"  -> New best model saved at epoch {global_epoch} (val_f1={best_val_f1:.4f})")

        if es.step(va["f1"]):
            print("  -> Early stopping triggered in stage 2")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, artifact_dir / "best_model_final.pth")

    # Optional ONNX export
    try:
        dummy = torch.randn(1, 3, img_size, img_size).to(DEVICE)
        model.eval()
        torch.onnx.export(
            model,
            dummy,
            str(artifact_dir / "best_model.onnx"),
            input_names=["input"],
            output_names=["logits"],
            opset_version=12
        )
    except Exception as e:
        print("ONNX export failed:", e)

    train_summary = {
        "model_name": model_name,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "img_size": img_size,
        "batch_size": batch_size,
        "train_size": len(train_df),
        "val_size": len(val_df),
        "test_size": len(test_df),
        "output_dir": str(run_dir)
    }
    with open(run_dir / "train_summary.json", "w") as f:
        json.dump(train_summary, f, indent=2)

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "history.csv", index=False)

    # Plots
    plt.figure(figsize=(10,5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.plot(history_df["epoch"], history_df["train_f1"], label="train_f1")
    plt.plot(history_df["epoch"], history_df["val_f1"], label="val_f1")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title(f"Training History - {model_name}")
    plt.tight_layout()
    plt.savefig(plot_dir / "training_history.png", dpi=200)
    plt.close()

    plt.figure(figsize=(8,5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title(f"Loss Curve - {model_name}")
    plt.tight_layout()
    plt.savefig(plot_dir / "loss_curve.png", dpi=200)
    plt.close()

    plt.figure(figsize=(8,5))
    plt.plot(history_df["epoch"], history_df["train_qwk"], label="train_qwk")
    plt.plot(history_df["epoch"], history_df["val_qwk"], label="val_qwk")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title(f"QWK Curve - {model_name}")
    plt.tight_layout()
    plt.savefig(plot_dir / "qwk_curve.png", dpi=200)
    plt.close()

    plt.figure(figsize=(8,5))
    plt.plot(history_df["epoch"], history_df["lr"], label="learning_rate")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title(f"Learning Rate - {model_name}")
    plt.tight_layout()
    plt.savefig(plot_dir / "learning_rate_curve.png", dpi=200)
    plt.close()

    # Confusion Matrix - Tensorboard
    try:
        import seaborn as sns
        import matplotlib.pyplot as plt

        # gunakan validation terakhir
        cm = va["confusion_matrix"] if "confusion_matrix" in va else None
        if cm is not None:
            fig, ax = plt.subplots()
            sns.heatmap(cm, annot=True, fmt="d", ax=ax)
            tb_writer.add_figure("Confusion Matrix", fig)
            plt.close(fig)
    except Exception as e:
        print("TensorBoard confusion matrix failed:", e)

    tb_writer.flush()
    tb_writer.close()

    if use_wandb and WANDB_AVAILABLE:
        wandb.finish()

    print("\n" + "=" * 80)
    print(f"TRAINING FINISHED: {model_name}")
    print(f"Best epoch     : {best_epoch}")
    print(f"Best val_f1    : {best_val_f1:.4f}")
    print(f"Saved to       : {run_dir}")
    print("=" * 80)

    return run_dir

In [ ]:
# Benchmark latency/FPS
def benchmark_latency(model, img_size=224, warmup=20, runs=50):
    model.eval()
    x = torch.randn(1, 3, img_size, img_size).to(DEVICE)

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(x)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start = time.time()
        for _ in range(runs):
            _ = model(x)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.time() - start

    latency = elapsed / runs
    fps = 1.0 / latency
    return latency, fps

## Evaluation

In [ ]:
# Evaluation
def evaluate_model(model_name, run_dir, img_size=224, batch_size=16, num_workers=2, dropout=0.35, amp=True):
    _, _, test_loader, _, train_df, val_df, test_df = make_loaders(
        df_all,
        img_size=img_size,
        batch_size=batch_size,
        num_workers=num_workers,
        imbalance_mode="weighted_loss"
    )

    model = build_model(model_name, num_classes=NUM_CLASSES, dropout=dropout).to(DEVICE)
    ckpt = torch.load(run_dir / "artifacts" / "best_model_final.pth", map_location=DEVICE)
    model.load_state_dict(ckpt)

    criterion = nn.CrossEntropyLoss()
    te = run_epoch(model, test_loader, criterion, optimizer=None, train=False, amp=amp)

    cm = confusion_matrix(te["y_true"], te["y_pred"], labels=[0,1,2,3,4])

    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(run_dir / "plots" / "confusion_matrix.png", dpi=200)
    plt.close()

    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_df.to_csv(run_dir / "confusion_matrix.csv")

    report = classification_report(
        te["y_true"], te["y_pred"],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )
    pd.DataFrame(report).transpose().to_csv(run_dir / "classification_report.csv")

    pred_df = pd.DataFrame({
        "filepath": te["paths"],
        "true_label": te["y_true"],
        "pred_label": te["y_pred"],
        "true_name": [CLASS_NAMES[i] for i in te["y_true"]],
        "pred_name": [CLASS_NAMES[i] for i in te["y_pred"]],
        "confidence": te["y_prob"].max(axis=1)
    })
    for i in range(NUM_CLASSES):
        pred_df[f"prob_{i}"] = te["y_prob"][:, i]

    pred_df.to_csv(run_dir / "predictions" / "test_predictions.csv", index=False)

    latency, fps = benchmark_latency(model, img_size=img_size)

    summary = {
        "model": model_name,
        "test_loss": te["loss"],
        "accuracy": te["accuracy"],
        "precision_weighted": te["precision"],
        "recall_weighted": te["recall"],
        "f1_weighted": te["f1"],
        "qwk": te["qwk"],
        "balanced_accuracy": te["balanced_accuracy"],
        "latency_sec_per_image": latency,
        "fps": fps,
        "num_test_images": len(test_df)
    }

    with open(run_dir / "evaluation_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"Important evaluation outputs saved to drive: {run_dir}")
    return summary

In [ ]:
# Qualitative results
def show_qualitative_samples(run_dir, n=8):
    pred_df = pd.read_csv(run_dir / "predictions" / "test_predictions.csv")

    correct_df = pred_df[pred_df["true_label"] == pred_df["pred_label"]]
    wrong_df = pred_df[pred_df["true_label"] != pred_df["pred_label"]]

    samples = []
    if len(correct_df) > 0:
        samples.append(("Correct Predictions", correct_df.sample(min(n, len(correct_df)), random_state=SEED)))
    if len(wrong_df) > 0:
        samples.append(("Wrong Predictions", wrong_df.sample(min(n, len(wrong_df)), random_state=SEED)))

    for title, subset in samples:
        cols = 4
        rows = math.ceil(len(subset) / cols)
        plt.figure(figsize=(16, 4 * rows))

        for i, (_, row) in enumerate(subset.iterrows(), 1):
            img = Image.open(row["filepath"]).convert("RGB")
            plt.subplot(rows, cols, i)
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"T: {row['true_name']}\nP: {row['pred_name']}\nConf: {row['confidence']:.3f}")

        plt.suptitle(title)
        plt.tight_layout()
        save_name = title.lower().replace(" ", "_") + ".png"
        plt.savefig(run_dir / "plots" / save_name, dpi=200)
        plt.show()
        plt.close()

In [ ]:
# Confidence rate
def plot_prediction_confidence(run_dir, num_samples=5):
    df = pd.read_csv(run_dir / "predictions" / "test_predictions.csv")
    samples = df.sample(min(num_samples, len(df)), random_state=SEED)

    for i, row in samples.iterrows():
        probs = [row[f"prob_{j}"] for j in range(5)]

        plt.figure(figsize=(6,4))
        plt.bar(CLASS_NAMES, probs)
        plt.xticks(rotation=30)
        plt.title(f"True: {row['true_name']} | Pred: {row['pred_name']} | Conf: {row['confidence']:.3f}")
        plt.ylabel("Probability")
        plt.tight_layout()

        save_path = run_dir / "plots" / f"confidence_bar_{i}.png"
        plt.savefig(save_path, dpi=200)
        plt.show()
        plt.close()

In [ ]:
# Model comparison
def compare_models(output_root=OUTPUT_ROOT):
    model_names = ["mobilenetv3", "efficientnet_b0", "swin_t", "effb0_swin_ensemble"]
    rows = []

    for m in model_names:
        f = Path(output_root) / m / "evaluation_summary.json"
        if f.exists():
            with open(f, "r") as fp:
                rows.append(json.load(fp))

    cmp_df = pd.DataFrame(rows)
    display(cmp_df)

    comparison_dir = Path(output_root) / "comparison"
    comparison_dir.mkdir(parents=True, exist_ok=True)

    cmp_df.to_csv(comparison_dir / "comparison_metrics.csv", index=False)

    for metric in ["accuracy", "f1_weighted", "qwk", "balanced_accuracy", "fps"]:
        if metric in cmp_df.columns:
            plt.figure(figsize=(8,4))
            sns.barplot(data=cmp_df, x="model", y=metric)
            plt.xticks(rotation=20)
            plt.title(metric)
            plt.tight_layout()
            plt.savefig(comparison_dir / f"{metric}_comparison.png", dpi=200)
            plt.show()
            plt.close()

    if {"f1_weighted", "balanced_accuracy", "fps"}.issubset(set(cmp_df.columns)):
        tmp = cmp_df.copy()
        tmp["deployment_score"] = (
            0.4 * tmp["f1_weighted"] +
            0.3 * tmp["balanced_accuracy"] +
            0.3 * (tmp["fps"] / tmp["fps"].max())
        )
        tmp = tmp.sort_values("deployment_score", ascending=False)
        tmp.to_csv(comparison_dir / "deployment_ranking.csv", index=False)

    with open(comparison_dir / "comparison_summary.json", "w") as f:
        json.dump(cmp_df.to_dict(orient="records"), f, indent=2)

    print("Comparison outputs saved to drive:", comparison_dir)
    return cmp_df

## Main

In [ ]:
run_dir_mobilenet = train_one_model(
    model_name="mobilenetv3",
    img_size=256,
    batch_size=32,
    num_workers=4,
    epochs_head=5,
    epochs_ft=10,
    lr_head=1e-3,
    lr_ft=2e-4,
    imbalance_mode="weighted_loss",
    use_wandb=False
)

evaluate_model("mobilenetv3", run_dir_mobilenet, img_size=256, batch_size=64)
show_qualitative_samples(run_dir_mobilenet, n=8)
plot_prediction_confidence(run_dir_mobilenet, num_samples=5)

In [ ]:
run_dir_eff = train_one_model(
    model_name="efficientnet_b0",
    img_size=300,
    batch_size=16,
    num_workers=4,
    epochs_head=5,
    epochs_ft=12,
    lr_head=1e-3,
    lr_ft=2e-4,
    imbalance_mode="weighted_loss",
    use_wandb=False
)

evaluate_model("efficientnet_b0", run_dir_eff, img_size=256, batch_size=64)
show_qualitative_samples(run_dir_eff, n=8)
plot_prediction_confidence(run_dir_eff, num_samples=5)

In [ ]:
run_dir_swin = train_one_model(
    model_name="swin_t",
    img_size=300,
    batch_size=16,
    num_workers=4,
    epochs_head=5,
    epochs_ft=12,
    lr_head=1e-3,
    lr_ft=1e-4,
    imbalance_mode="weighted_loss",
    use_wandb=False
)

evaluate_model("swin_t", run_dir_swin, img_size=256, batch_size=64)
show_qualitative_samples(run_dir_swin, n=8)
plot_prediction_confidence(run_dir_swin, num_samples=5)

In [ ]:
run_dir_ens = train_one_model(
    model_name="effb0_swin_ensemble",
    img_size=300,
    batch_size=16,
    num_workers=4,
    epochs_head=5,
    epochs_ft=12,
    lr_head=1e-3,
    lr_ft=1e-4,
    imbalance_mode="weighted_loss",
    unfreeze_mode="all",
    use_wandb=False
)

evaluate_model("effb0_swin_ensemble", run_dir_ens, img_size=300, batch_size=32)
show_qualitative_samples(run_dir_ens, n=8)
plot_prediction_confidence(run_dir_ens, num_samples=5)

In [ ]:
# Compare all models

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

def compare_models_with_confusion_matrices(
    output_root=OUTPUT_ROOT,
    model_names=None,
    class_names=CLASS_NAMES
):
    if model_names is None:
        # model_names = ["mobilenetv3", "efficientnet_b0", "swin_t", "effb0_swin_ensemble"]
        model_names = ["efficientnet_b0", "swin_t", "effb0_swin_ensemble"]

    rows = []
    comparison_dir = Path(output_root) / "model-comparison-ensemble"
    comparison_dir.mkdir(parents=True, exist_ok=True)

    available_models = []

    for model_name in model_names:
        model_dir = Path(output_root) / model_name
        eval_path = model_dir / "evaluation_summary.json"
        cm_path = model_dir / "confusion_matrix.csv"

        if not eval_path.exists():
            print(f"[SKIP] evaluation_summary.json tidak ditemukan untuk {model_name}")
            continue

        with open(eval_path, "r") as f:
            summary = json.load(f)

        # Classification metrices
        rows.append({
            "model": summary.get("model", model_name),
            "accuracy": summary.get("accuracy", None),
            "precision_weighted": summary.get("precision_weighted", None),
            "recall_weighted": summary.get("recall_weighted", None),
            "f1_weighted": summary.get("f1_weighted", None),
            "qwk": summary.get("qwk", None),
            "balanced_accuracy": summary.get("balanced_accuracy", None),
            "test_loss": summary.get("test_loss", None),
            "fps": summary.get("fps", None),
            "latency_sec_per_image": summary.get("latency_sec_per_image", None),
            "num_test_images": summary.get("num_test_images", None),
        })

        if cm_path.exists():
            available_models.append(model_name)

    if len(rows) == 0:
        print("Tidak ada hasil evaluasi model yang ditemukan.")
        return None

    cmp_df = pd.DataFrame(rows)
    cmp_df = cmp_df.sort_values(by="f1_weighted", ascending=False).reset_index(drop=True)

    print("\n=== MODEL COMPARISON TABLE ===")
    display(cmp_df)

    cmp_df.to_csv(comparison_dir / "comparison_metrics.csv", index=False)

    # Save JSON
    with open(comparison_dir / "comparison_metrics.json", "w") as f:
        json.dump(cmp_df.to_dict(orient="records"), f, indent=2)

    # Bar chart
    metrics_to_plot = [
        "accuracy",
        "precision_weighted",
        "recall_weighted",
        "f1_weighted",
        "qwk",
        "balanced_accuracy",
        "fps"
    ]

    for metric in metrics_to_plot:
        if metric in cmp_df.columns and cmp_df[metric].notna().any():
            plt.figure(figsize=(8, 6))
            sns.barplot(data=cmp_df, x="model", y=metric)
            plt.xticks(rotation=20)
            plt.title(f"{metric} Comparison")
            plt.tight_layout()
            plt.savefig(comparison_dir / f"{metric}_comparison.png", dpi=200)
            plt.show()
            plt.close()

    # Show Confusion matrix for each model
    print("\n=== CONFUSION MATRIX EACH MODEL ===")
    for model_name in available_models:
        cm_path = Path(output_root) / model_name / "confusion_matrix.csv"

        if not cm_path.exists():
            print(f"[SKIP] confusion_matrix.csv tidak ditemukan untuk {model_name}")
            continue

        cm_df = pd.read_csv(cm_path, index_col=0)

        plt.figure(figsize=(8, 6))
        sns.heatmap(
            cm_df,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names
        )
        plt.title(f"Confusion Matrix - {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(comparison_dir / f"{model_name}_confusion_matrix_comparison.png", dpi=200)
        plt.show()
        plt.close()


    # deployment score
    if {"f1_weighted", "balanced_accuracy", "fps"}.issubset(set(cmp_df.columns)):
        tmp = cmp_df.copy()
        tmp["deployment_score"] = (
            0.4 * tmp["f1_weighted"].fillna(0) +
            0.3 * tmp["balanced_accuracy"].fillna(0) +
            0.3 * (tmp["fps"].fillna(0) / max(1e-8, tmp["fps"].fillna(0).max()))
        )
        tmp = tmp.sort_values("deployment_score", ascending=False).reset_index(drop=True)

        print("\n=== DEPLOYMENT RANKING ===")
        display(tmp[[
            "model", "f1_weighted", "balanced_accuracy", "fps", "deployment_score"
        ]])

        tmp.to_csv(comparison_dir / "deployment_ranking_with_cm.csv", index=False)

    print(f"\nAll comparison outputs saved to: {comparison_dir}")
    return cmp_df

In [ ]:
# Compare
cmp_df_with_cm = compare_models_with_confusion_matrices()
cmp_df_with_cm

In [ ]:
# TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/DR_Severity-1/outputs_dr